# Optimization and Machine Learning Prediction of Tensile and Compressive Strengths for Additively Manufactured ABS Automotive Components
**Course:** Introduction to Digital Manufacturing
**Reference paper:** Munshi, Kulkarni & Yargatti, *Next Materials* 10 (2026) 101420

This notebook walks through the results produced by `python run_project.py` (run it first from the project root). The last section trains the best model live and explains a recipe of your choice with SHAP.

In [ ]:
import os, sys, json
import pandas as pd
import numpy as np
from IPython.display import Image, display

ROOT = os.path.abspath('..')
sys.path.append(os.path.join(ROOT, 'src'))
FIG = os.path.join(ROOT, 'outputs', 'figures')
TAB = os.path.join(ROOT, 'outputs', 'tables')
pd.set_option('display.width', 200, 'display.max_columns', 30)

from data_loader import load_raw_data, get_param_bounds, get_param_levels, FEATURE_COLUMNS, TARGET_COLUMNS
summary = json.load(open(os.path.join(TAB, 'summary_key_results.json')))
print('Best model:', summary['best_model_pretty'])

## 1. Dataset (383 samples, 5 FDM parameters, 2 strengths)

In [ ]:
df = load_raw_data()
print('Shape:', df.shape)
print('Parameter window:', get_param_bounds(df))
print('Tested levels   :', get_param_levels(df))
pd.read_csv(os.path.join(TAB, 'dataset_statistics.csv'), index_col=0)

## 2. Pearson correlation

In [ ]:
display(Image(os.path.join(FIG, 'pearson_correlation_heatmap.png'), width=650))

## 3. Model ranking and all 9 metrics (Review 1 feedback #1)

In [ ]:
pd.read_csv(os.path.join(TAB, 'model_ranking.csv'), index_col=0).round(4)

In [ ]:
ext = pd.read_csv(os.path.join(TAB, 'extended_metrics_holdout.csv'))
ext.round(4)

In [ ]:
display(Image(os.path.join(FIG, 'metrics_r2_comparison.png'), width=900))
display(Image(os.path.join(FIG, 'metrics_error_heatmap.png'), width=800))

## 4. Comparison with the published paper (Tables 4 and 7)

In [ ]:
pd.read_csv(os.path.join(TAB, 'Table7_R2_comparison_paper_vs_ours.csv'))

In [ ]:
pd.read_csv(os.path.join(TAB, 'Table4_benchmark_comparison_paper_vs_ours.csv'))

## 5. Explainability with SHAP, in MPa (Review 1 feedback #2)

In [ ]:
display(pd.read_csv(os.path.join(TAB, 'shap_global_importance.csv')))
display(Image(os.path.join(FIG, 'shap_summary_tensile.png'), width=750))
display(Image(os.path.join(FIG, 'shap_waterfall_tensile_optimum.png'), width=750))

In [ ]:
pd.read_csv(os.path.join(TAB, 'shap_local_explanations.csv'))

## 6. Optimal parameters, sensitivity and case studies (Review 1 feedback #3)

In [ ]:
pd.read_csv(os.path.join(TAB, 'optimal_parameters.csv'))

In [ ]:
display(pd.read_csv(os.path.join(TAB, 'sensitivity_analysis.csv')))
display(Image(os.path.join(FIG, 'sensitivity_response_curves.png'), width=950))

In [ ]:
display(pd.read_csv(os.path.join(TAB, 'automotive_case_study_recommendations.csv')))
display(Image(os.path.join(FIG, 'pareto_frontier_automotive.png'), width=750))

## 7. Taguchi L15 combinations of the paper (Table 3)

In [ ]:
pd.read_csv(os.path.join(TAB, 'taguchi_L15_predictions.csv'))

## 8. Try your own recipe
This trains the best model on all 383 samples, predicts the strengths of a recipe and explains the prediction in MPa. Edit `recipe` and re-run the cell.

In [ ]:
from advanced_models import get_advanced_models
from baseline_models import get_baseline_models
from shap_explainability import ShapExplainerMPa
from optimizer import worst_case_predict, estimate_relative_print_time

models = {**get_baseline_models(), **get_advanced_models()}
model = models[summary['best_model']]
model.fit(df[FEATURE_COLUMNS].values, df[TARGET_COLUMNS].values)

# Nozzle °C, Bed °C, Speed mm/s, Layer mm, Infill %
recipe = np.array([[230, 80, 30, 0.2, 100]], dtype=float)

pred = model.predict(recipe)[0]
wc = worst_case_predict(model, recipe)[0]
print(f'Predicted tensile     : {pred[0]:.2f} MPa (worst case within tolerance {wc[0]:.2f})')
print(f'Predicted compressive : {pred[1]:.2f} MPa (worst case within tolerance {wc[1]:.2f})')
print(f'Relative print time   : {estimate_relative_print_time(recipe[0, 2], recipe[0, 3], recipe[0, 4]):.1f}')

explainer = ShapExplainerMPa(model, df[FEATURE_COLUMNS].values)
sv, base = explainer.explain(recipe, 0)
print(f'\nTensile = {base:.2f} (average)', end='')
for name, v in zip(FEATURE_COLUMNS, sv[0]):
    print(f' {v:+.2f} ({name})', end='')
print(f' = {base + sv[0].sum():.2f} MPa')